<a href="https://colab.research.google.com/github/Trangnguyen1402/AAI2025/blob/2026Fall/house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

# Data source:
# Realtor.com housing data provided in realtor-data.csv
# File: realtor-data.csv
#
# If you downloaded this file from another website, add that website URL here.

# Load the Realtor dataset
df = pd.read_csv("realtor-data.csv")

# Use sold homes in California
df = df[
    (df["status"] == "sold") &
    (df["state"] == "California")
].copy()

# Keep rows with the required information
df = df[
    df["price"].notna() &
    df["house_size"].notna() &
    df["city"].notna()
]

# Remove invalid values
df = df[
    (df["price"] > 0) &
    (df["house_size"] > 0)
]

# Remove extreme values and possible data errors
for column in ["price", "house_size"]:
    lower_limit = df[column].quantile(0.01)
    upper_limit = df[column].quantile(0.99)

    df = df[
        df[column].between(lower_limit, upper_limit)
    ]

# Create the three location categories required by the assignment
# These are simplified categories based on California cities.
downtown_cities = {
    "Los Angeles",
    "San Francisco",
    "San Jose",
    "Oakland",
    "San Diego",
    "Sacramento"
}

rural_cities = {
    "Blythe",
    "Barstow",
    "Needles",
    "King City",
    "Willits",
    "Madera",
    "Hanford",
    "Visalia",
    "El Centro",
    "Mojave",
    "Paso Robles"
}

def assign_location(city):
    if city in downtown_cities:
        return "Downtown"
    elif city in rural_cities:
        return "Rural"
    else:
        return "Suburb"

df["location"] = df["city"].apply(assign_location)

# Select exactly 200 records while keeping all three location categories
downtown_data = df[
    df["location"] == "Downtown"
].sample(n=80, random_state=42)

suburb_data = df[
    df["location"] == "Suburb"
].sample(n=80, random_state=42)

rural_data = df[
    df["location"] == "Rural"
].sample(n=40, random_state=42)

df = pd.concat([
    downtown_data,
    suburb_data,
    rural_data
])

# Shuffle the final dataset
df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Rename the square footage column
df = df.rename(columns={
    "house_size": "square_footage"
})

print(f"Number of records used: {len(df)}")
print("\nLocation counts:")
print(df["location"].value_counts())

# Save the adjusted 200-record dataset
df.to_csv(
    "realtor_sample_200.csv",
    index=False
)

# Features and target
X = df[[
    "square_footage",
    "location"
]]

y = df["price"]

# One-hot encode location
# Downtown is used as the reference location.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "location",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["location"]
        )
    ],
    remainder="passthrough"
)

# Create the linear regression pipeline
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Test the model
predictions = model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    predictions
)

r2 = r2_score(
    y_test,
    predictions
)

print(f"\nMean Absolute Error: ${mae:,.2f}")
print(f"R-squared Score: {r2:.3f}")

# Predict a 2,000-square-foot house in Downtown
new_house = pd.DataFrame({
    "square_footage": [2000],
    "location": ["Downtown"]
})

predicted_price = model.predict(
    new_house
)

print(
    f"\nPredicted price for a 2,000-square-foot "
    f"house in Downtown: ${predicted_price[0]:,.2f}"
)

# Display model coefficients
feature_names = (
    model.named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    model.named_steps["regressor"]
    .coef_
)

print("\nModel Coefficients:")

for feature, coefficient in zip(
    feature_names,
    coefficients
):
    print(f"{feature}: ${coefficient:,.2f}")

# Explanation required by the checklist
print("\nCoefficient Explanation:")
print(
    "- The square footage coefficient estimates how much "
    "the predicted price changes for one additional square foot."
)
print(
    "- The Rural and Suburb coefficients compare those locations "
    "with Downtown, which is the reference location."
)

Number of records used: 200

Location counts:
location
Suburb      80
Downtown    80
Rural       40
Name: count, dtype: int64

Mean Absolute Error: $373,210.14
R-squared Score: 0.213

Predicted price for a 2,000-square-foot house in Downtown: $1,228,655.66

Model Coefficients:
location__location_Rural: $-642,382.50
location__location_Suburb: $-162,076.41
remainder__square_footage: $543.24

Coefficient Explanation:
- The square footage coefficient estimates how much the predicted price changes for one additional square foot.
- The Rural and Suburb coefficients compare those locations with Downtown, which is the reference location.
